# 03 - Feature Engineering

**Objetivo:** Construir el conjunto de características (features) y la variable objetivo que alimentarán los modelos econométricos y de ML en el notebook `04_modeling.ipynb`.

**Input:** `data/processed/dataset_modelo.csv` — 408 observaciones mensuales (ene-1992 → dic-2025), 12 variables, sin nulos.

**Output:** `data/processed/features.csv` — dataset listo para modelado con el target `inflation_mom` y ~110 variables derivadas.

**Metodología:**
- Target: variación porcentual mensual del CPI (`inflation_mom = CPI.pct_change() * 100`).
- Features derivados por variable: rezagos (1, 3, 6, 12 meses), promedios móviles (3, 6, 12), variaciones MoM y YoY.
- Tratamiento especial de `gold_price` (log-transform, dado que no alcanza estacionariedad con primera diferencia; ver informe estadístico, sección 8).
- Variables temporales (mes, trimestre) con codificación cíclica.
- Todas las transformaciones respetan la secuencialidad temporal: ningún feature utiliza información futura.

## 1. Setup

In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

# Reproducibilidad
np.random.seed(42)

# Rutas relativas a la raíz del repositorio
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
PATH_INPUT = ROOT / 'data' / 'processed' / 'dataset_modelo.csv'
PATH_OUTPUT = ROOT / 'data' / 'processed' / 'features.csv'

# Constantes del diseño de features
LAGS = [1, 3, 6, 12]
MOVING_AVG_WINDOWS = [3, 6, 12]
HORIZONS = [1, 3, 6, 12]  # se usará en el notebook 04; lo declaramos aquí por trazabilidad

print(f"Input : {PATH_INPUT.relative_to(ROOT)}")
print(f"Output: {PATH_OUTPUT.relative_to(ROOT)}")

Input : data/processed/dataset_modelo.csv
Output: data/processed/features.csv


## 2. Cargar dataset integrado

In [2]:
df = pd.read_csv(PATH_INPUT, parse_dates=['date'])
df = df.sort_values('date').reset_index(drop=True)
df = df.set_index('date')

print(f"Shape: {df.shape}")
print(f"Período: {df.index.min():%Y-%m} → {df.index.max():%Y-%m}")
print(f"Nulos: {df.isna().sum().sum()}")
df.head()

Shape: (408, 12)
Período: 1992-01 → 2025-12
Nulos: 0


,cpi,fed_rate,oil_price,unemployment,industrial_production,money_supply_m2,retail_sales,capacity_utilization,treasury_10y,ppi,consumer_sentiment,gold_price
date,,,,,,,,,,,,
1992-01-01,138.3,4.03,18.785455,7.3,61.4616,3381.2,115095.0,79.3278,7.03,115.6,67.5,354.45
1992-02-01,138.6,4.06,19.012500,7.4,61.8955,3400.0,114855.0,79.7329,7.34,116.0,68.8,353.91
1992-03-01,139.1,3.98,18.921818,7.4,62.4232,3403.9,114052.0,80.2464,7.54,116.1,76.0,344.34
1992-04-01,139.4,3.73,20.230000,7.4,62.8970,3399.7,114721.0,80.6802,7.48,116.3,77.2,338.62
1992-05-01,139.7,3.82,20.975500,7.6,63.1040,3398.6,114868.0,80.7640,7.39,117.2,79.2,337.24


## 3. Variable objetivo: inflación mensual

La variable objetivo se define como la **variación porcentual mes a mes del CPI**:

$$y_t = \frac{CPI_t - CPI_{t-1}}{CPI_{t-1}} \times 100$$

**Unidad:** puntos porcentuales (%).

**Motivación:**
- El CPI en niveles es I(1) (no estacionario). Predecirlo directamente infla artificialmente las métricas: un modelo trivial que copia el valor anterior obtiene R² > 0.99 sin información real.
- La transformación a variación porcentual elimina la tendencia y produce una serie aproximadamente estacionaria, adecuada para comparación justa entre modelos.
- Es la definición estándar en la literatura (Medeiros et al., 2021; Nguyen et al., 2023; FMI, 2024).

La primera observación queda como NaN (no hay mes anterior) y se descarta en el ensamblaje.

In [3]:
target = df['cpi'].pct_change() * 100
target.name = 'inflation_mom'

print(f"Target: {target.name}")
print(f"Observaciones: {target.notna().sum()} válidas / {len(target)} totales (primera es NaN por pct_change)")
print()
print("Estadísticas descriptivas de la inflación mensual:")
print(target.describe().round(3))

Target: inflation_mom
Observaciones: 407 válidas / 408 totales (primera es NaN por pct_change)

Estadísticas descriptivas de la inflación mensual:
count    407.000
mean       0.211
std        0.270
min       -1.771
25%        0.088
50%        0.213
75%        0.323
max        1.377
Name: inflation_mom, dtype: float64


## 4. Ingeniería de características

Se generan cuatro familias de features sobre las 12 variables macroeconómicas. Todas las transformaciones usan únicamente datos hasta el instante *t-1* para evitar fuga de información temporal.

### 4.1 Rezagos (lags)

Captura la inercia de cada indicador: el valor observado *k* meses atrás es predictor del estado actual.

**Diseño:** lags de 1, 3, 6 y 12 meses. El lag-12 permite capturar patrones estacionales anuales sin necesidad de dummies de mes.

**Efecto:** por cada variable original se generan 4 features. Con 12 variables, total = 48 features de rezago.

In [4]:
def add_lags(df_in: pd.DataFrame, lags: list) -> pd.DataFrame:
    """Genera columnas de rezago para cada variable.

    Nomenclatura: <variable>_lag<k>
    """
    out = {}
    for col in df_in.columns:
        for k in lags:
            out[f'{col}_lag{k}'] = df_in[col].shift(k)
    return pd.DataFrame(out, index=df_in.index)

df_lags = add_lags(df, LAGS)
print(f"Features de rezago generados: {df_lags.shape[1]}")
print(f"Ejemplo (primeras columnas): {list(df_lags.columns[:6])}")

Features de rezago generados: 48
Ejemplo (primeras columnas): ['cpi_lag1', 'cpi_lag3', 'cpi_lag6', 'cpi_lag12', 'fed_rate_lag1', 'fed_rate_lag3']


### 4.2 Promedios móviles

Suavizan ruido de corto plazo y revelan tendencias de mediano plazo.

**Diseño:** ventanas de 3, 6 y 12 meses. Se calculan **sobre el valor rezagado (t-1)** para no incluir el periodo actual (eso introduciría look-ahead bias al predecir *t*).

**Efecto:** 3 ventanas × 12 variables = 36 features.

In [5]:
def add_moving_averages(df_in: pd.DataFrame, windows: list) -> pd.DataFrame:
    """Genera promedios móviles sobre el valor rezagado 1 periodo.

    Se aplica .shift(1) antes de .rolling() para garantizar que el MA
    en el mes t solo use información hasta el mes t-1.
    """
    out = {}
    for col in df_in.columns:
        shifted = df_in[col].shift(1)
        for w in windows:
            out[f'{col}_ma{w}'] = shifted.rolling(window=w, min_periods=w).mean()
    return pd.DataFrame(out, index=df_in.index)

df_ma = add_moving_averages(df, MOVING_AVG_WINDOWS)
print(f"Features de promedio móvil generados: {df_ma.shape[1]}")
print(f"Ejemplo: {list(df_ma.columns[:6])}")

Features de promedio móvil generados: 36
Ejemplo: ['cpi_ma3', 'cpi_ma6', 'cpi_ma12', 'fed_rate_ma3', 'fed_rate_ma6', 'fed_rate_ma12']


### 4.3 Variaciones porcentuales (MoM, YoY)

Capturan la dinámica de cambio más que el nivel. Complementan los rezagos al expresar la serie en términos estacionarios.

**Diseño:**
- **MoM** (month-over-month): $\Delta_{1m} = (x_t - x_{t-1}) / x_{t-1}$
- **YoY** (year-over-year): $\Delta_{12m} = (x_t - x_{t-12}) / x_{t-12}$

Ambas se aplican sobre el valor rezagado 1 mes para mantener la regla "solo información hasta *t-1*".

**Efecto:** 2 tipos × 12 variables = 24 features.

In [6]:
def add_pct_changes(df_in: pd.DataFrame) -> pd.DataFrame:
    """Variaciones porcentuales MoM y YoY sobre valor rezagado."""
    out = {}
    for col in df_in.columns:
        shifted = df_in[col].shift(1)
        out[f'{col}_mom'] = shifted.pct_change(periods=1) * 100
        out[f'{col}_yoy'] = shifted.pct_change(periods=12) * 100
    return pd.DataFrame(out, index=df_in.index)

df_pct = add_pct_changes(df)
print(f"Features de variaciones generados: {df_pct.shape[1]}")
print(f"Ejemplo: {list(df_pct.columns[:6])}")

Features de variaciones generados: 24
Ejemplo: ['cpi_mom', 'cpi_yoy', 'fed_rate_mom', 'fed_rate_yoy', 'oil_price_mom', 'oil_price_yoy']


### 4.4 Transformación logarítmica del precio del oro

El test ADF aplicado en el EDA mostró que `gold_price` **no alcanza estacionariedad ni con primera diferencia** (ADF de 1ª diff = 0.668, p > 0.05; ver informe estadístico, sección 8). Esto se debe al rally exponencial 2020–2025 que amplifica la varianza.

**Tratamiento:** aplicar `log(gold_price)` y luego diferencia. La transformación logarítmica estabiliza la varianza antes de la diferencia.

Se añaden como features adicionales: `gold_price_log` rezagado y su diferencia mensual.

In [7]:
gold_log = np.log(df['gold_price'])
df_gold = pd.DataFrame({
    'gold_price_log_lag1':   gold_log.shift(1),
    'gold_price_log_diff1':  gold_log.diff(1).shift(1),
    'gold_price_log_diff12': gold_log.diff(12).shift(1),
}, index=df.index)

print(f"Features especiales de gold_price: {df_gold.shape[1]}")
print(df_gold.describe().round(4))

Features especiales de gold_price: 3
       gold_price_log_lag1  gold_price_log_diff1  gold_price_log_diff12
count             407.0000              406.0000               395.0000
mean                6.6481                0.0060                 0.0685
std                 0.7534                0.0350                 0.1453
min                 5.5455               -0.1248                -0.3215
25%                 5.9275               -0.0149                -0.0259
50%                 6.8269                0.0016                 0.0549
75%                 7.2575                0.0279                 0.1706
max                 8.3156                0.1601                 0.4706


### 4.5 Variables temporales

Complementan las macroeconómicas con información del calendario.

**Diseño:**
- Mes codificado en forma cíclica (`sin`, `cos`) para que diciembre y enero queden "cerca" en el espacio de features, algo que no ocurre con un entero 1–12 ni con one-hot.
- Trimestre como one-hot (4 columnas).

**Efecto:** 2 (seno/coseno) + 4 (trimestre) = 6 features temporales. No llevan lag porque son conocidas de antemano.

In [8]:
month = df.index.month
quarter = df.index.quarter

df_time = pd.DataFrame({
    'month_sin': np.sin(2 * np.pi * month / 12),
    'month_cos': np.cos(2 * np.pi * month / 12),
}, index=df.index)

for q in [1, 2, 3, 4]:
    df_time[f'quarter_{q}'] = (quarter == q).astype(int)

print(f"Features temporales generados: {df_time.shape[1]}")
df_time.head()

Features temporales generados: 6


,month_sin,month_cos,quarter_1,quarter_2,quarter_3,quarter_4
date,,,,,,
1992-01-01,0.500000,8.660254e-01,1,0,0,0
1992-02-01,0.866025,5.000000e-01,1,0,0,0
1992-03-01,1.000000,6.123234e-17,1,0,0,0
1992-04-01,0.866025,-5.000000e-01,0,1,0,0
1992-05-01,0.500000,-8.660254e-01,0,1,0,0


## 5. Ensamblaje del dataset final

Se concatenan todos los bloques de features junto con el target. Se eliminan las primeras observaciones que contienen NaN por efecto de los lags de 12 meses (pérdida de ~12 filas).

In [9]:
features = pd.concat([df_lags, df_ma, df_pct, df_gold, df_time], axis=1)
dataset = features.copy()
dataset['inflation_mom'] = target

print(f"Total features: {features.shape[1]}")
print(f"Dataset ensamblado (con target): {dataset.shape}")
print(f"Rango: {dataset.index.min():%Y-%m} → {dataset.index.max():%Y-%m}")
print()
print(f"Nulos antes de dropna: {dataset.isna().sum().sum()}")
print(f"Filas con algún NaN: {dataset.isna().any(axis=1).sum()}")

Total features: 117
Dataset ensamblado (con target): (408, 118)
Rango: 1992-01 → 2025-12

Nulos antes de dropna: 713
Filas con algún NaN: 13


In [10]:
dataset_clean = dataset.dropna()

print(f"Dataset limpio: {dataset_clean.shape}")
print(f"Rango efectivo: {dataset_clean.index.min():%Y-%m} → {dataset_clean.index.max():%Y-%m}")
print(f"Observaciones perdidas: {len(dataset) - len(dataset_clean)} (esperado: 12 por lag12/YoY)")

Dataset limpio: (395, 118)
Rango efectivo: 1993-02 → 2025-12
Observaciones perdidas: 13 (esperado: 12 por lag12/YoY)


## 6. Validación del dataset

Tres verificaciones críticas antes de pasar a modelado:

1. **Sin NaN residuales** — cualquier nulo restante contaminaría el entrenamiento.
2. **Sin data leakage temporal** — el target en *t* no debe correlacionarse perfectamente con features construidos con datos ≥ *t*.
3. **Correlaciones razonables** — features con correlación = ±1.0 con el target indican un error en la construcción.

In [11]:
assert dataset_clean.isna().sum().sum() == 0, "Hay nulos residuales"
print(f"✓ Sin nulos residuales: {dataset_clean.isna().sum().sum()}")

corr_with_target = dataset_clean.drop(columns='inflation_mom').corrwith(dataset_clean['inflation_mom']).abs()
max_corr = corr_with_target.max()
top_corr = corr_with_target.sort_values(ascending=False).head(10)
print(f"\n✓ Correlación máxima feature↔target: {max_corr:.4f}  (alerta si ≥ 0.99)")
print("\nTop 10 features más correlacionados con inflation_mom:")
print(top_corr.round(3).to_string())

✓ Sin nulos residuales: 0

✓ Correlación máxima feature↔target: 0.4676  (alerta si ≥ 0.99)

Top 10 features más correlacionados con inflation_mom:
oil_price_mom       0.468
ppi_mom             0.462
cpi_mom             0.458
oil_price_yoy       0.308
treasury_10y_mom    0.287
retail_sales_yoy    0.264
ppi_yoy             0.254
cpi_yoy             0.227
treasury_10y_yoy    0.222
retail_sales_mom    0.197


In [12]:
diffs_days = dataset_clean.index.to_series().diff().dt.days.dropna().unique()
assert set(diffs_days).issubset({28, 29, 30, 31}), f"Gaps temporales inesperados: {diffs_days}"
print(f"✓ Índice mensual continuo: diffs únicos (días) = {sorted(diffs_days)}")

✓ Índice mensual continuo: diffs únicos (días) = [np.float64(28.0), np.float64(29.0), np.float64(30.0), np.float64(31.0)]


## 7. Guardar dataset

El archivo de salida contiene el target y todos los features alineados por fecha. Será el insumo de `04_modeling.ipynb`.

In [13]:
dataset_clean.to_csv(PATH_OUTPUT, index=True)

print("=" * 60)
print("DATASET DE FEATURES GUARDADO")
print("=" * 60)
print(f"Archivo : {PATH_OUTPUT.relative_to(ROOT)}")
print(f"Filas   : {len(dataset_clean)}")
print(f"Columnas: {dataset_clean.shape[1]} ({dataset_clean.shape[1] - 1} features + 1 target)")
print(f"Período : {dataset_clean.index.min():%Y-%m} → {dataset_clean.index.max():%Y-%m}")
print()
print("Desglose de features:")
print(f"  • Rezagos           : {df_lags.shape[1]}")
print(f"  • Promedios móviles : {df_ma.shape[1]}")
print(f"  • Variaciones %     : {df_pct.shape[1]}")
print(f"  • Gold especial     : {df_gold.shape[1]}")
print(f"  • Temporales        : {df_time.shape[1]}")
print(f"  • TOTAL             : {features.shape[1]}")

DATASET DE FEATURES GUARDADO
Archivo : data/processed/features.csv
Filas   : 395
Columnas: 118 (117 features + 1 target)
Período : 1993-02 → 2025-12

Desglose de features:
  • Rezagos           : 48
  • Promedios móviles : 36
  • Variaciones %     : 24
  • Gold especial     : 3
  • Temporales        : 6
  • TOTAL             : 117


## Resumen

**Target:** `inflation_mom = CPI.pct_change() * 100` (variación porcentual mensual del CPI, en puntos porcentuales).

**Feature set (~114 columnas):**

| Familia | Descripción | # |
|---|---|---|
| Rezagos | Lags 1, 3, 6, 12 por variable | 48 |
| Promedios móviles | Ventanas 3, 6, 12 sobre lag-1 | 36 |
| Variaciones % | MoM y YoY por variable | 24 |
| Gold especial | log y diferencias log | 3 |
| Temporales | Mes (sin/cos), trimestre (one-hot) | 6 |

**Decisiones clave:**
- Todos los features usan solo información hasta *t-1* (sin look-ahead).
- Gold recibe tratamiento logarítmico por no ser estacionario ni en 1ª diferencia.
- Mes codificado en forma cíclica para preservar continuidad diciembre ↔ enero.

**Output:** `data/processed/features.csv`, ~396 observaciones × ~115 columnas.

**Próximo paso:** `04_modeling.ipynb` — ARIMA(1,1,1) benchmark + Elastic Net + Random Forest + XGBoost con validación walk-forward expandible, horizontes h = 1, 3, 6, 12.